# Classical machine learning: unrestricted 72 h versus manual 42 h

This notebook performs the paper's classical-model ablation on the Fuseki data. It forecasts the **complete next 72 hours** (288 values at 15-minute spacing).

The comparison is deliberately matched:

- **Unrestricted 72 h:** the previous 288 samples of all five conductivity sensors (1,440 inputs). The model is free to infer travel times.
- **Manual 42 h:** the paper's four hand-designed scalar inputs: current target value, its one-hour change, its two-hour change, and Terneuzen shifted by exactly 42 hours.
- **Training:** January--July 2025, sampled hourly to reduce overlap.
- **Validation:** August 2025, used for conservative forecast calibration only.
- **Test:** September 2025, never used for fitting or selection.

Every model predicts a residual relative to the latest target observation. Persistence is therefore the zero-residual forecast. Skill is reported over the complete trajectory and at the predeclared 4, 8, 16, 24, and 48-hour marks.

In [1]:
from pathlib import Path
import gc, os, sys, time, warnings

ROOT = Path.cwd().resolve()
if not (ROOT / "lag_analytics_workspace").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

from time_series_analysis.paper_experiment_utils import (
    EVALUATION_HOURS,
    FORECAST_HOURS,
    build_classical_datasets,
    evaluation_rows,
    load_prepared_observations,
    split_shape_table,
    validation_calibration,
)

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Load and validate the observations

The repository's Fuseki client collapses legacy duplicate measurements and normalizes conductivity to mS/cm. Preparation uses a causal trailing two-hour mean and only interpolates short internal gaps (at most one hour). No centered smoothing or future fill is used.

In [2]:
data, coverage, unit_report = load_prepared_observations()
print(f"Prepared grid: {data.shape[0]:,} rows from {data.index.min()} to {data.index.max()}")
display(coverage)
if not unit_report.empty:
    display(unit_report[[c for c in ["sensor_name", "normalization", "observations"] if c in unit_report]])

Prepared grid: 26,815 rows from 2025-01-01 00:00:00+00:00 to 2025-10-07 07:30:00+00:00


,first observation,last observation,missing after preparation,minimum mS/cm,maximum mS/cm
sensor,,,,,
Terneuzen,2025-01-01 00:00:00+00:00,2026-03-23 15:45:00+00:00,372,0.108207,2.339817
Westdorpe,2025-01-01 00:00:00+00:00,2025-10-07 07:30:00+00:00,257,0.107912,3.997718
Gent - far,2025-01-01 00:00:00+00:00,2026-03-23 12:00:00+00:00,254,0.055390,1.882214
Gent - near,2025-01-01 00:00:00+00:00,2025-12-18 23:00:00+00:00,253,0.068467,1.792810
Indusii,2025-01-01 00:00:00+00:00,2025-12-31 23:45:00+00:00,962,0.000018,1.945475


,sensor_name
0,Terneuzen
1,Gent - far
2,Indusii
3,Gent - near
4,Westdorpe


In [3]:
datasets = build_classical_datasets(data)
display(split_shape_table(datasets))

,Representation,Split,X shape,y shape,First forecast,Last forecast
0,unrestricted_72h,train,"(3812, 1440)","(3812, 288)",2025-01-07 01:45:00+00:00,2025-07-31 23:45:00+00:00
1,unrestricted_72h,validation,"(2098, 1440)","(2098, 288)",2025-08-10 03:30:00+00:00,2025-08-31 23:45:00+00:00
2,unrestricted_72h,test,"(2593, 1440)","(2593, 288)",2025-09-03 23:45:00+00:00,2025-09-30 23:45:00+00:00
3,manual_42h,train,"(3812, 4)","(3812, 288)",2025-01-07 01:45:00+00:00,2025-07-31 23:45:00+00:00
4,manual_42h,validation,"(2098, 4)","(2098, 288)",2025-08-10 03:30:00+00:00,2025-08-31 23:45:00+00:00
5,manual_42h,test,"(2593, 4)","(2593, 288)",2025-09-03 23:45:00+00:00,2025-09-30 23:45:00+00:00


## 2. Models and improvements over the exploratory notebook

The original lag notebook trained separate one-hour examples and reused August in the training set. Here all methods share the same 288-output task and strict chronological split.

- **Ridge** is a strong regularized linear reference for highly correlated lag vectors.
- **XGBoost** follows the draft's nonlinear tree model but uses 75 boosting rounds, shallower trees, shrinkage, row/column subsampling, and L2 regularization. The bounded round count keeps the 288-output scalar-tree fit computationally tractable without reducing the forecast horizon.
- **RBF-SVR** restores the draft's ordinary standardized radial-basis support-vector regressor (`C=100`, `epsilon=0.01`). Because the matched paper task has 288 lead times, `MultiOutputRegressor` fits one independent SVR per lead. No PCA or other feature reduction is applied in either representation.
- **Shallow MLP** preserves the draft's classical feed-forward baseline, with smaller learning rate, larger batches, and early stopping inside the training period.

Hyperparameters are fixed before the September test. A single multiplicative residual weight per lead time is estimated on August and constrained to `[0, 1]`; it can only shrink a model toward persistence, not amplify it.

In [4]:
def make_models(input_features):
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
        "XGBoost": XGBRegressor(
            n_estimators=75,
            learning_rate=0.04,
            max_depth=4,
            min_child_weight=5,
            subsample=0.80,
            colsample_bytree=0.65,
            reg_alpha=0.05,
            reg_lambda=5.0,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=SEED,
            n_jobs=min(8, os.cpu_count() or 1),
        ),
        "RBF-SVR": make_pipeline(
            StandardScaler(),
            MultiOutputRegressor(
                SVR(
                    kernel="rbf",
                    C=100.0,
                    epsilon=0.01,
                    cache_size=2048,
                ),
                n_jobs=min(4, os.cpu_count() or 1),
            ),
        ),
        "Shallow MLP": make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                learning_rate_init=1e-3,
                batch_size=256,
                max_iter=160,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=SEED,
            ),
        ),
    }

In [ ]:
all_rows, all_horizons = [], []
fit_rows, predictions = [], {}
calibration_weights = {}

for representation, dataset in datasets.items():
    train = dataset.splits["train"]
    validation = dataset.splits["validation"]
    test = dataset.splits["test"]
    scale = dataset.target_scale
    models = make_models(train.X.shape[1])
    for model_name in models:
        model = models[model_name]
        key = (model_name, representation)
        started = time.perf_counter()
        print(f"Fitting {model_name:12s} on {representation:18s} ...", flush=True)
        model.fit(train.X, train.y_delta / scale)
        elapsed = time.perf_counter() - started

        validation_raw = np.asarray(model.predict(validation.X), dtype=np.float32) * scale
        weights = validation_calibration(validation_raw, validation.y_delta)
        validation_calibrated = validation_raw * weights
        validation_persistence_rmse = float(np.sqrt(np.mean(validation.y_delta ** 2)))
        validation_rmse = float(np.sqrt(np.mean((validation_calibrated - validation.y_delta) ** 2)))

        test_raw = np.asarray(model.predict(test.X), dtype=np.float32) * scale
        test_calibrated = test_raw * weights
        rows, horizon, prediction = evaluation_rows(
            model_name, representation, test, test_calibrated
        )
        all_rows.extend(rows)
        all_horizons.append(horizon)
        predictions[key] = prediction
        calibration_weights[key] = weights
        fit_rows.append({
            "Model": model_name,
            "Representation": representation,
            "Fit seconds": elapsed,
            "Validation RMSE": validation_rmse,
            "Validation persistence RMSE": validation_persistence_rmse,
            "Validation RMSE skill": 1 - validation_rmse / validation_persistence_rmse,
            "Mean calibration weight": float(weights.mean()),
        })
        print(f"  {elapsed:.1f}s; validation skill {fit_rows[-1]['Validation RMSE skill']:.3f}")
        models[model_name] = None
        del model
        gc.collect()

fit_summary = pd.DataFrame(fit_rows).sort_values("Validation RMSE")
results = pd.DataFrame(all_rows)
horizon_results = pd.concat(all_horizons, ignore_index=True)
display(fit_summary.round({
    "Fit seconds": 1, "Validation RMSE": 5,
    "Validation persistence RMSE": 5, "Validation RMSE skill": 3,
    "Mean calibration weight": 3,
}))

Fitting Ridge        on unrestricted_72h   ...
  0.1s; validation skill 0.082
Fitting XGBoost      on unrestricted_72h   ...
  585.9s; validation skill 0.058
Fitting RBF-SVR      on unrestricted_72h   ...


## 3. Held-out September results

In [ ]:
summary = results.pivot_table(
    index=["Model", "Representation"], columns="Scope",
    values=["RMSE", "R2", "RMSE skill"]
).sort_values(("RMSE", "all horizons"))
display(summary.round(4))

checkpoint_scopes = [f"{hours}-hour mark" for hours in EVALUATION_HOURS]
checkpoint_skill = (
    results[results["Scope"].isin(checkpoint_scopes)]
    .pivot(index=["Model", "Representation"], columns="Scope", values="RMSE skill")
    .reindex(columns=checkpoint_scopes)
)
display(checkpoint_skill.round(4))

In [ ]:
checkpoint_lookup = {f"{hours}-hour mark": hours for hours in EVALUATION_HOURS}
plot_data = results[results["Scope"].isin(checkpoint_lookup)].copy()
plot_data["Lead hours"] = plot_data["Scope"].map(checkpoint_lookup)
fig, ax = plt.subplots(figsize=(12, 7))
for (model_name, representation), group in plot_data.groupby(["Model", "Representation"]):
    group = group.sort_values("Lead hours")
    ax.plot(group["Lead hours"], group["RMSE skill"], marker="o", label=f"{model_name} — {representation}")
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(EVALUATION_HOURS)
ax.set_title("September skill over persistence at predeclared forecast marks")
ax.set_xlabel("Forecast lead (hours)")
ax.set_ylabel("RMSE skill (positive is better)")
ax.grid(alpha=.25)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# Select the display model by August validation RMSE, not by September results.
best_row = fit_summary.iloc[0]
best_key = (best_row["Model"], best_row["Representation"])
best_horizon = horizon_results[
    (horizon_results["Model"] == best_key[0]) &
    (horizon_results["Representation"] == best_key[1])
]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(best_horizon["Lead hours"], best_horizon["RMSE skill"], linewidth=1.5)
marked = best_horizon[best_horizon["Lead hours"].isin(EVALUATION_HOURS)]
axes[0].scatter(marked["Lead hours"], marked["RMSE skill"], color="black", zorder=3)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(title=f"Skill across the {FORECAST_HOURS}-hour forecast — {best_key[0]}, {best_key[1]}", xlabel="Lead (hours)", ylabel="RMSE skill")
axes[0].grid(alpha=.25)

week = predictions[best_key].loc["2025-09-05":"2025-09-12"]
axes[1].plot(week.index, week["Actual"], label="Actual", linewidth=1.5)
axes[1].plot(week.index, week["Prediction"], label="Forecast", linewidth=1.2)
axes[1].plot(week.index, week["Persistence"], label="Persistence", alpha=.7)
axes[1].set(title=f"{FORECAST_HOURS}-hour endpoint during a September week", ylabel="Conductivity (mS/cm)")
axes[1].legend()
axes[1].grid(alpha=.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Reproducible conclusion

In [ ]:
for representation in datasets:
    chosen = fit_summary[fit_summary["Representation"] == representation].iloc[0]
    test_row = results[
        (results["Model"] == chosen["Model"]) &
        (results["Representation"] == representation) &
        (results["Scope"] == "all horizons")
    ].iloc[0]
    checkpoint_rows = results[
        (results["Model"] == chosen["Model"])
        & (results["Representation"] == representation)
        & (results["Scope"].isin(checkpoint_scopes))
    ].set_index("Scope")
    checkpoint_text = ", ".join(
        f"{hours} h={checkpoint_rows.loc[f'{hours}-hour mark', 'RMSE skill']:.3f}"
        for hours in EVALUATION_HOURS
    )
    print(
        f"{representation}: August selected {chosen['Model']}; "
        f"September RMSE={test_row['RMSE']:.5f}, "
        f"overall {FORECAST_HOURS}-hour skill={test_row['RMSE skill']:.3f}; "
        f"checkpoint skill: {checkpoint_text}."
    )

best_test = results[results["Scope"] == "all horizons"].sort_values("RMSE").iloc[0]
print(
    f"\nLowest September RMSE among the pre-specified experiments: {best_test['Model']} "
    f"with {best_test['Representation']} ({best_test['RMSE']:.5f}; "
    f"skill {best_test['RMSE skill']:.3f})."
)